In [ ]:
import pandas as pd

BASE_DIR = "/Users/ipekgezer"

CY_PATH   = f"{BASE_DIR}/country_year_women_ratio.csv"
VDEM_PATH = f"{BASE_DIR}/V-Dem-CY-Full+Others-v15.csv"          
OUT_PATH  = f"{BASE_DIR}/country_year_women_ratio_vdem_multi.csv"


cy = pd.read_csv(CY_PATH)
print("CY rows:", len(cy), "cols:", len(cy.columns))

# V-DEM FEATURES 
vdem_feats = [
    "v2x_polyarchy",
    "v2x_libdem",
    "v2x_egaldem",
    "v2x_civlib",
    "v2x_clpol",
    "v2x_clphy",
    "v2fsuffrage",
    "v2lgqugen",
    "v2lgfemleg",
]

vdem_cols = ["country_text_id", "year"] + vdem_feats

vdem = pd.read_csv(VDEM_PATH, usecols=vdem_cols, low_memory=False)
print("VDEM rows:", len(vdem), "cols:", len(vdem.columns))

# merge (country-year)
merged = cy.merge(
    vdem,
    left_on=["country", "year"],
    right_on=["country_text_id", "year"],
    how="left",
    suffixes=("", "_vdem")
)

# Drop duplicate key column from V-Dem side
if "country_text_id" in merged.columns:
    merged = merged.drop(columns=["country_text_id"])

# ========= DIAGNOSTICS =========
# match rate for one key feature (change if you want)
key_feat = "v2x_polyarchy"
key_col = key_feat if key_feat in merged.columns else f"{key_feat}_vdem"

match_rate = merged[key_col].notna().mean()
print("Merged rows:", len(merged))
print(f"V-Dem match rate ({key_col} not null):", match_rate)

# ========= SAVE =========
merged.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("Saved:", OUT_PATH)
